In [ ]:
# 2026-10-19
import pandas as pd
from csae_pyutils import gsheet_to_df
from tqdm import tqdm
from apis_core.apis_metainfo.models import Collection, Uri
from apis_core.apis_entities.models import Person, Place
from apis_core.apis_vocabularies.models import PersonPlaceRelation, ProfessionType
from apis_core.apis_relations.models import PersonPlace

In [ ]:
col = Collection.objects.filter(name__startswith="Brahms Online").first()
domain = "brahms-online"
born_in_rel = PersonPlaceRelation.objects.get(name="geboren in")
died_in_rel = PersonPlaceRelation.objects.get(name="gestorben in")
born_in_rel, died_in_rel, col, domain



In [ ]:
df = gsheet_to_df("1J3mo2D7-MoQ5Xm-rOJxSE9ga0_ze1BR9vAuCMYYzbTo")

In [ ]:
prof_lookup = {
    "Sänger_in": "Sänger/Sängerin",
    "Pianist_in": "Pianist/Pianistin",
    "Violinist_in": "Violinist/Violinistin",
    "Bratschist_in": "Bratschist/Bratschistin",
}

In [ ]:
df.head(3)

In [ ]:
for i, row in tqdm(df.iterrows(), total=len(df)):
    tmp_id = row["URIs"].split(";")[0].strip().split("/")[-1]
    pmb_uri = f"https://brahms-online.oeaw.ac.at/personenregister-id-{tmp_id}.html"
    pmb_uri, _ = Uri.objects.get_or_create(uri=pmb_uri, domain=domain)
    entity = pmb_uri.entity
    if entity:
        person = Person.objects.get(id=entity.id)
        pass
    else:
        item = {
            "name": row["nachname"][:254] if pd.notna(row["nachname"]) else "",
            "first_name": row["vorname"][:254] if pd.notna(row["vorname"]) else "",
        }
        entity = Person.objects.create(**item)
        person = Person.objects.get(id=entity.id)
        pmb_uri.entity = entity
        pmb_uri.save()
    try:
        for y in row["profession"].split(";"):
            prof_label = y.strip()
            try:
                prof_norm = prof_lookup[prof_label]
                prof_obj = ProfessionType.objects.get(name=prof_norm)
                person.profession.add(prof_obj)
            except KeyError:
                continue
    except AttributeError:
        pass
    if pd.isna(row["birth-when-iso"]):
        pass
    else:
        entity.start_date_written = row["birth-when-iso"]
    if pd.isna(row["death-when-iso"]):
        pass
    else:
        entity.end_date_written = row["death-when-iso"]
    entity.collection.add(col)
    entity.save()
    if not pd.isna(row["birth-place"]):
        place = Place.objects.filter(name=row["birth-place"]).first()
        place = place if place is not None else Place.objects.get_or_create(name=row["birth-place"])[0]
        PersonPlace.objects.get_or_create(
            related_person=person,
            related_place=place,
            start_date_written=row["birth-when-iso"] if pd.notna(row["birth-when-iso"]) else "",
            end_date_written=row["birth-when-iso"] if pd.notna(row["birth-when-iso"]) else "",
            relation_type=born_in_rel
        )
    if not pd.isna(row["death-place"]):
            place = Place.objects.filter(name=row["death-place"]).first()
            place = place if place is not None else Place.objects.get_or_create(name=row["death-place"])[0]
            PersonPlace.objects.get_or_create(
                related_person=person,
                related_place=place,
                start_date_written=row["death-when-iso"] if pd.notna(row["death-when-iso"]) else "",
                end_date_written=row["death-when-iso"] if pd.notna(row["death-when-iso"]) else "",
                relation_type=died_in_rel
            )
   